In [1]:
#Install
!pip install requests beautifulsoup4
!pip install selenium webdriver-manager

In [2]:
# Imports
import requests
from bs4 import BeautifulSoup
import os
import time

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/120.0.0.0 Safari/537.36"
}
BASE_URL = "https://huggingface.co"
print("Working directory:", os.getcwd())

Working directory: C:\Users\HP\Machine Learning


In [3]:
# Navigate with Selenium, fallback to hardcoded list
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

def get_vision_models_selenium():
    options = webdriver.ChromeOptions()
    options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")

    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options
    )

    driver.get("https://huggingface.co/docs/transformers/index")
    time.sleep(5)  # wait for JS to fully render

    # Try multiple selectors to find sidebar links
    models = []
    all_links = driver.find_elements(By.TAG_NAME, "a")

    in_vision = False
    for link in all_links:
        try:
            href = link.get_attribute("href") or ""
            text = link.text.strip()

            if not href or not text:
                continue

            # Detect vision section header
            if "vision" in text.lower() and len(text) < 40:
                in_vision = True
                print(f"  Found section: {text}")
                continue

            # Stop at next section (e.g. "Multimodal Models", "Audio Models")
            if in_vision and len(text) > 3 and len(text) < 40:
                if any(x in text.lower() for x in ["multimodal", "audio", "text", "reinforcement", "time series"]):
                    print(f"  Stopping at: {text}")
                    break

            if in_vision and "/model_doc/" in href:
                models.append((text, href))
        except:
            continue

    driver.quit()
    return models

# Try Selenium first
print("Attempting dynamic navigation with Selenium...")
vision_models = get_vision_models_selenium()
print(f"Selenium found: {len(vision_models)} models")

# Fallback to hardcoded list if Selenium found nothing
if len(vision_models) == 0:
    print("\nSelenium found 0 — using known vision model list as fallback...")
    VISION_MODEL_NAMES = [
        "beit", "bit", "conditional_detr", "convnext", "convnextv2",
        "cvt", "deformable_detr", "deit", "deta", "detr", "dinat",
        "dinov2", "efficientformer", "efficientnet", "glpn",
        "imagegpt", "levit", "mask2former", "maskformer", "mobilenet_v1",
        "mobilenet_v2", "mobilevit", "mobilevitv2", "nat",
        "poolformer", "pvt", "regnet", "resnet", "segformer",
        "swiftformer", "swin", "swin2sr", "swinv2", "table_transformer",
        "timesformer", "upernet", "van", "videomae", "vit",
        "vit_hybrid", "vit_mae", "vit_msn", "vitdet", "vivit"
    ]
    vision_models = [
        (name, f"{BASE_URL}/docs/transformers/en/model_doc/{name}")
        for name in VISION_MODEL_NAMES
    ]

print(f"\nTotal vision models to process: {len(vision_models)}")
for name, url in vision_models:
    print(f"  {name:30s}  →  {url}")

Attempting dynamic navigation with Selenium...
Selenium found: 0 models

Selenium found 0 — using known vision model list as fallback...

Total vision models to process: 44
  beit                            →  https://huggingface.co/docs/transformers/en/model_doc/beit
  bit                             →  https://huggingface.co/docs/transformers/en/model_doc/bit
  conditional_detr                →  https://huggingface.co/docs/transformers/en/model_doc/conditional_detr
  convnext                        →  https://huggingface.co/docs/transformers/en/model_doc/convnext
  convnextv2                      →  https://huggingface.co/docs/transformers/en/model_doc/convnextv2
  cvt                             →  https://huggingface.co/docs/transformers/en/model_doc/cvt
  deformable_detr                 →  https://huggingface.co/docs/transformers/en/model_doc/deformable_detr
  deit                            →  https://huggingface.co/docs/transformers/en/model_doc/deit
  deta                      

In [4]:
# Create main folder
MAIN_FOLDER = os.path.join(os.getcwd(), "Vision Models")
os.makedirs(MAIN_FOLDER, exist_ok=True)
print("Main folder created at:", MAIN_FOLDER)

Main folder created at: C:\Users\HP\Machine Learning\Vision Models


In [5]:
# Scrape and save each model
for model_name, model_url in vision_models:

    folder_name  = model_name.replace("/", "_").strip()
    model_folder = os.path.join(MAIN_FOLDER, folder_name)
    os.makedirs(model_folder, exist_ok=True)

    try:
        page = requests.get(model_url, headers=HEADERS, timeout=15)

        if page.status_code != 200:
            print(f"Skipped (HTTP {page.status_code}): {model_name}")
            continue

        m_soup = BeautifulSoup(page.text, "html.parser")

        for tag in m_soup(["script", "style", "nav", "footer", "header"]):
            tag.decompose()

        main = m_soup.find("main") or m_soup.find("article") or m_soup
        text = main.get_text(separator="\n", strip=True)

        file_path = os.path.join(model_folder, "documentation.txt")
        with open(file_path, "w", encoding="utf-8") as f:
            f.write(f"Model : {model_name}\n")
            f.write(f"URL   : {model_url}\n")
            f.write("=" * 60 + "\n\n")
            f.write(text)

        size_kb = os.path.getsize(file_path) / 1024
        print(f"Saved: {model_name:30s}  ({size_kb:.1f} KB)")

    except Exception as e:
        print(f"Error — {model_name}: {e}")

    time.sleep(0.5)

print("\nDone! All models saved under:", MAIN_FOLDER)

Saved: beit                            (48.9 KB)
Saved: bit                             (23.6 KB)
Saved: conditional_detr                (58.1 KB)
Saved: convnext                        (23.3 KB)
Saved: convnextv2                      (14.6 KB)
Saved: cvt                             (15.2 KB)
Saved: deformable_detr                 (45.0 KB)
Saved: deit                            (38.0 KB)
Saved: deta                            (0.3 KB)
Saved: detr                            (85.8 KB)
Saved: dinat                           (19.5 KB)
Saved: dinov2                          (19.9 KB)
Saved: efficientformer                 (0.3 KB)
Saved: efficientnet                    (25.3 KB)
Saved: glpn                            (22.4 KB)
Saved: imagegpt                        (44.5 KB)
Saved: levit                           (30.8 KB)
Saved: mask2former                     (57.7 KB)
Saved: maskformer                      (56.3 KB)
Saved: mobilenet_v1                    (23.3 KB)
Saved: mobilenet_v2   

In [6]:
# Verify
print(f"\nContents of 'Vision Models':\n")
count = 0
for folder in sorted(os.listdir(MAIN_FOLDER)):
    folder_path = os.path.join(MAIN_FOLDER, folder)
    if os.path.isdir(folder_path):
        files = os.listdir(folder_path)
        total_size = sum(os.path.getsize(os.path.join(folder_path, f)) for f in files)
        print(f"  {folder}/  →  {len(files)} file(s),  {total_size/1024:.1f} KB")
        count += 1

print(f"\nTotal subfolders: {count}")


Contents of 'Vision Models':

  beit/  →  1 file(s),  48.9 KB
  bit/  →  1 file(s),  23.6 KB
  conditional_detr/  →  1 file(s),  58.1 KB
  convnext/  →  1 file(s),  23.3 KB
  convnextv2/  →  1 file(s),  14.6 KB
  cvt/  →  1 file(s),  15.2 KB
  deformable_detr/  →  1 file(s),  45.0 KB
  deit/  →  1 file(s),  38.0 KB
  deta/  →  1 file(s),  0.3 KB
  detr/  →  1 file(s),  85.8 KB
  dinat/  →  1 file(s),  19.5 KB
  dinov2/  →  1 file(s),  19.9 KB
  efficientformer/  →  1 file(s),  0.3 KB
  efficientnet/  →  1 file(s),  25.3 KB
  glpn/  →  1 file(s),  22.4 KB
  imagegpt/  →  1 file(s),  44.5 KB
  levit/  →  1 file(s),  30.8 KB
  mask2former/  →  1 file(s),  57.7 KB
  maskformer/  →  1 file(s),  56.3 KB
  mobilenet_v1/  →  1 file(s),  23.3 KB
  mobilenet_v2/  →  1 file(s),  33.6 KB
  mobilevit/  →  1 file(s),  31.7 KB
  mobilevitv2/  →  1 file(s),  21.0 KB
  nat/  →  1 file(s),  0.3 KB
  poolformer/  →  1 file(s),  25.0 KB
  pvt/  →  1 file(s),  24.2 KB
  regnet/  →  1 file(s),  13.9 KB
  r